Copyright (c) Microsoft Corporation. All rights reserved. 

Licensed under the MIT License.

# Use FLAML to Tune OpenAI Models

FLAML offers a cost-effective hyperparameter optimization technique [EcoOptiGen](https://arxiv.org/abs/2303.04673) for tuning Large Language Models. Our study finds that tuning hyperparameters can significantly improve the utility of LLMs.

In this notebook, we tune OpenAI models for code generation. We use [the HumanEval benchmark](https://huggingface.co/datasets/openai_humaneval) released by OpenAI for synthesizing programs from docstrings. 

## Requirements

FLAML requires `Python>=3.7`. To run this notebook example, please install flaml with the `[synapse,openai]` option:
```bash
pip install flaml[synapse,openai]
```

In [ ]:
%pip install "flaml[synapse,openai]@https://automlsaeastus.blob.core.windows.net/releases/FLAML-latest-py3-none-any.whl" datasets

StatementMeta(, f382d186-b7b8-4fd0-a694-6748bd86ff2a, 8, Finished, Available)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.1/275.1 kB 2.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.6/474.6 kB 27.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 93.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 24.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 34.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 78.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 kB 45.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.5/212.5 kB 81.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 84.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.7/78.7 kB 38.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.2/147.2 kB 60.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 22.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━

In [ ]:
# Setup OpenAI
import openai
from synapse.ml.mlflow import get_mlflow_env_config

mlflow_env_configs = get_mlflow_env_config()
access_token = mlflow_env_configs.driver_aad_token
baseurl = mlflow_env_configs.workload_endpoint + "cognitive/openai/"

openai.api_key = mlflow_env_configs.driver_aad_token
openai.api_base = baseurl
openai.api_type = "azure_ad"
openai.api_version = "2023-03-15-preview"

StatementMeta(, f382d186-b7b8-4fd0-a694-6748bd86ff2a, 10, Finished, Available)

## Load dataset

First, we load the humaneval dataset. The dataset contains 164 examples. We use the first 20 for tuning the generation hyperparameters and the remaining for evaluation. In each example, the "prompt" is the prompt string for eliciting the code generation (renamed into "definition"), "test" is the Python code for unit test for the example, and "entry_point" is the function name to be tested.

In [ ]:
import datasets

seed = 41
data = datasets.load_dataset("openai_humaneval")["test"].shuffle(seed=seed)
n_tune_data = 20
tune_data = [
    {
        "definition": data[x]["prompt"],
        "test": data[x]["test"],
        "entry_point": data[x]["entry_point"],
    }
    for x in range(n_tune_data)
]
test_data = [
    {
        "definition": data[x]["prompt"],
        "test": data[x]["test"],
        "entry_point": data[x]["entry_point"],
    }
    for x in range(n_tune_data, len(data))
]


StatementMeta(, f382d186-b7b8-4fd0-a694-6748bd86ff2a, 12, Finished, Available)

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Dataset openai_humaneval downloaded and prepared to /home/trusted-service-user/.cache/huggingface/datasets/openai_humaneval/openai_humaneval/1.0.0/2955cebd73602e828fa8c0a424c594e5fab4ec863b316ca98f3d8fdb6a626e75. Subsequent calls will reuse this data.


  0%|          | 0/1 [00:00<?, ?it/s]

StatementMeta(, f382d186-b7b8-4fd0-a694-6748bd86ff2a, 22, Finished, Available)

Check a tuning example:

In [ ]:
print(tune_data[1]["definition"])

StatementMeta(, f382d186-b7b8-4fd0-a694-6748bd86ff2a, 13, Finished, Available)


def compare(game,guess):
    """I think we all remember that feeling when the result of some long-awaited
    event is finally known. The feelings and thoughts you have at that moment are
    definitely worth noting down and comparing.
    Your task is to determine if a person correctly guessed the results of a number of matches.
    You are given two arrays of scores and guesses of equal length, where each index shows a match. 
    Return an array of the same length denoting how far off each guess was. If they have guessed correctly,
    the value is 0, and if not, the value is the absolute difference between the guess and the score.
    
    
    example:

    compare([1,2,3,4,5,1],[1,2,3,4,2,-2]) -> [0,0,0,0,3,3]
    compare([0,5,0,0,0,4],[4,1,1,0,0,-2]) -> [4,4,1,0,0,6]
    """



Here is one example of the unit test code for verifying the correctness of the generated code:

In [ ]:
print(tune_data[1]["test"])

StatementMeta(, f382d186-b7b8-4fd0-a694-6748bd86ff2a, 14, Finished, Available)

def check(candidate):

    # Check some simple cases
    assert candidate([1,2,3,4,5,1],[1,2,3,4,2,-2])==[0,0,0,0,3,3], "This prints if this assert fails 1 (good for debugging!)"
    assert candidate([0,0,0,0,0,0],[0,0,0,0,0,0])==[0,0,0,0,0,0], "This prints if this assert fails 1 (good for debugging!)"
    assert candidate([1,2,3],[-1,-2,-3])==[2,4,6], "This prints if this assert fails 1 (good for debugging!)"
    assert candidate([1,2,3,5],[-1,2,3,4])==[2,0,0,1], "This prints if this assert fails 1 (good for debugging!)"

    # Check some edge cases that are easy to work out by hand.
    assert True, "This prints if this assert fails 2 (also good for debugging!)"




## Define Success Metric

Before we start tuning, we need to define the success metric we want to optimize. For each code generation task, we can use the model to generate multiple candidates, and then select one from them. If the final selected response can pass a unit test, we consider the task as successfully solved. Then we can define the mean success rate of a collection of tasks.

In [ ]:
from functools import partial
from flaml.autogen.code_utils import eval_function_completions, generate_assertions

eval_with_generated_assertions = partial(
    eval_function_completions,
    assertions=generate_assertions,
    use_docker=False,
    # Please set use_docker=True if you have docker available to run the generated code.
    # Using docker is safer than running the generated code directly.
)


StatementMeta(, f382d186-b7b8-4fd0-a694-6748bd86ff2a, 15, Finished, Available)

This function will first generate assertion statements for each problem. Then, it uses the assertions to select the generated responses.

## Use the tuning data to find a good configuration

### Import the oai and tune subpackages from flaml.

FLAML has provided an API for hyperparameter optimization of OpenAI models: `oai.Completion.tune` and to make a request with the tuned config: `oai.Completion.create`. First, we import oai from flaml:

In [ ]:
from flaml import oai, tune

StatementMeta(, f382d186-b7b8-4fd0-a694-6748bd86ff2a, 16, Finished, Available)

For (local) reproducibility and cost efficiency, we cache responses from OpenAI.

In [ ]:
oai.Completion.set_cache(seed)

StatementMeta(, f382d186-b7b8-4fd0-a694-6748bd86ff2a, 17, Finished, Available)

This will create a disk cache in ".cache/{seed}". You can change `cache_path` in `set_cache()`. The cache for different seeds are stored separately.

### Perform tuning

The tuning will take a while to finish, depending on the optimization budget. The tuning will be performed under the specified optimization budgets.

* `inference_budget` is the target average inference budget per instance in the benchmark. For example, 0.02 means the target inference budget is 0.02 dollars, which translates to 1000 tokens (input + output combined) if the text Davinci model is used.
* `optimization_budget` is the total budget allowed to perform the tuning. For example, 5 means 5 dollars are allowed in total, which translates to 250K tokens for the text Davinci model.
* `num_sumples` is the number of different hyperparameter configurations which is allowed to try. The tuning will stop after either num_samples trials or after optimization_budget dollars spent, whichever happens first. -1 means no hard restriction in the number of trials and the actual number is decided by `optimization_budget`.

Users can specify tuning data, optimization metric, optimization mode, evaluation function, search spaces etc.. The default search space is:

```python
default_search_space = {
    "model": tune.choice([
        "text-ada-001",
        "text-babbage-001",
        "text-davinci-003",
        "gpt-3.5-turbo",
        "gpt-4",
    ]),
    "temperature_or_top_p": tune.choice(
        [
            {"temperature": tune.uniform(0, 1)},
            {"top_p": tune.uniform(0, 1)},
        ]
    ),
    "max_tokens": tune.lograndint(50, 1000),
    "n": tune.randint(1, 100),
    "prompt": "{prompt}",
}
```

The default search space can be overridden by users' input.
For example, the following code specifies three choices for the prompt and two choices of stop sequences. For hyperparameters which don't appear in users' input, the default search space will be used. If you don't have access to gpt-4 or would like to modify the choice of models, you can provide a different search space for model.

In [ ]:
config, analysis = oai.Completion.tune(
    data=tune_data,  # the data for tuning
    metric="success",  # the metric to optimize
    mode="max",  # the optimization mode
    eval_func=eval_with_generated_assertions,  # the evaluation function to return the success metrics
    # log_file_name="logs/humaneval.log",  # the log file name
    inference_budget=0.05,  # the inference budget (dollar per instance)
    optimization_budget=1,  # the optimization budget (dollar in total)
    # num_samples can further limit the number of trials for different hyperparameter configurations;
    # -1 means decided by the optimization budget only
    num_samples=-1,
    model=tune.choice([
        "text-ada-001",
        "text-davinci-003",
        "gpt-35-turbo",
        "gpt-4",
    ]),
    prompt=[
        "{definition}",
        "# Python 3{definition}",
        "Complete the following Python function:{definition}",
    ],  # the prompt templates to choose from
    stop=[["\nclass", "\ndef", "\nif", "\nprint"], None],  # the stop sequences
)


StatementMeta(, f382d186-b7b8-4fd0-a694-6748bd86ff2a, 18, Finished, Available)

[I 2023-05-30 13:34:29,798] A new study created in memory with name: optuna
[I 2023-05-30 13:34:29,807] A new study created in memory with name: optuna


No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune
No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune
[flaml.tune.tune: 05-30 13:34:31] {807} INFO - trial 1 config: {'prompt': 1, 'stop': 0, 'subspace': {'model': 'text-ada-001', 'max_tokens': 148, 'temperature_or_top_p': {'top_p': 0.755486898036596}, 'n': 27}}
[flaml.tune.tune: 05-30 13:36:02] {205} INFO - result: {'index_selected': 26.0, 'succeed_assertions': 0.0, 'success': 0.0, 'gen_cost': 0.00046370000000000005, 'assertions': 'assert vowels_count("abcde") == 2\nassert vowels_count("ACEDY") =

### Output tuning results

After the tuning, we can print out the config and the result found by FLAML:

In [ ]:
print("optimized config", config)
print("best result on tuning data", analysis.best_result)

StatementMeta(, f382d186-b7b8-4fd0-a694-6748bd86ff2a, 19, Finished, Available)

optimized config {'prompt': '# Python 3{definition}', 'stop': ['\nclass', '\ndef', '\nif', '\nprint'], 'model': 'text-davinci-003', 'max_tokens': 148, 'n': 27, 'top_p': 0.755486898036596}
best result on tuning data {'index_selected': 4.1, 'succeed_assertions': 0.9, 'success': 0.7, 'gen_cost': 0.00046370000000000005, 'assertions': 'assert vowels_count("abcde") == 2\nassert vowels_count("ACEDY") == 3', 'total_cost': 0.8625920000000001, 'cost': 0.8510400000000001, 'inference_cost': 0.042002000000000005, 'training_iteration': 0, 'config': {'prompt': 1, 'stop': 0, 'subspace': {'model': 'text-davinci-003', 'max_tokens': 148, 'temperature_or_top_p': {'top_p': 0.755486898036596}, 'n': 27}}, 'config/prompt': 1, 'config/stop': 0, 'config/subspace': {'model': 'text-davinci-003', 'max_tokens': 148, 'temperature_or_top_p': {'top_p': 0.755486898036596}, 'n': 27}, 'experiment_tag': 'exp', 'time_total_s': 92.07447385787964}


### Make a request with the tuned config

We can apply the tuned config on the request for an example task:

In [ ]:
response = oai.Completion.create(context=tune_data[1], **config)
print(response)
print(eval_with_generated_assertions(oai.Completion.extract_text(response), **tune_data[1]))


StatementMeta(, f382d186-b7b8-4fd0-a694-6748bd86ff2a, 20, Finished, Available)

{
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "text": "    result = []\n    for i in range(len(game)):\n        result.append(abs(game[i] - guess[i]))\n    return result"
    },
    {
      "finish_reason": "stop",
      "index": 1,
      "logprobs": null,
      "text": "    results = []\n    for i in range(len(game)):\n        diff = abs(game[i] - guess[i])\n        results.append(diff)\n    return results"
    },
    {
      "finish_reason": "stop",
      "index": 2,
      "logprobs": null,
      "text": "    result = []\n    for i in range(len(game)):\n        result.append(abs(game[i] - guess[i]))\n    return result"
    },
    {
      "finish_reason": "stop",
      "index": 3,
      "logprobs": null,
      "text": "    results = []\n    for i in range(len(game)):\n        if game[i] == guess[i]:\n            results.append(0)\n        else:\n            results.append(abs(game[i] - guess[i]))\n    return results"
    },
    {

### Evaluate the success rate on the test data

You can use flaml's `oai.Completion.test` to evaluate the performance of an entire dataset with the tuned config. The following code will take a while to evaluate all the 144 test data instances. The cost is about $6 if you uncomment it and run it.

In [ ]:
# result = oai.Completion.test(test_data, **config)
# print("performance on test data with the tuned config:", result)

StatementMeta(, f382d186-b7b8-4fd0-a694-6748bd86ff2a, 21, Finished, Available)

The result will vary with the inference budget and optimization budget.
